In [1]:
from typing import TypedDict, Required, NotRequired

class RAGPipelineConfig(TypedDict, total=False):
    # Always required: retrieval and LLM configuration
    vector_db: Required[str]        # "chromadb", "pinecone", etc.
    llm_model: Required[str]        # "gpt-4", "gemini-pro", "llama-3-70b"
    
    # Optional: tuning and caching
    temperature: NotRequired[float] # Defaults to 0.1
    cache_results: NotRequired[bool]

def init_rag_pipeline(cfg: RAGPipelineConfig):
    print(f"Connecting to {cfg['vector_db']} with {cfg['llm_model']}...")
    print(f"Temperature = {cfg.get('temperature', 0.1)}")

# Example usage
init_rag_pipeline({"vector_db": "chromadb", "llm_model": "llama-3-70b"}) 


Connecting to chromadb with llama-3-70b...
Temperature = 0.1


In [ ]:
from typing import Annotated, get_type_hints

class Range:
    def __init__(self, min_val: float, max_val: float):
        self.min_val, self.max_val = min_val, max_val

LearningRate = Annotated[float, Range(1e-6, 1e-1)]

def train_model(lr: LearningRate, epochs: int):
    hints = get_type_hints(train_model, include_extras=True)
    lr_range = next(m for m in hints["lr"].__metadata__ if isinstance(m, Range))
    if not (lr_range.min_val <= lr <= lr_range.max_val):
        raise ValueError(f"Learning rate must be between {lr_range.min_val} and {lr_range.max_val}")
    print(f"Training with lr={lr} for {epochs} epochs...")

train_model(0.001, 10)  # OK
# train_model(0.5, 10)  # Will raise ValueError


Training with lr=0.001 for 10 epochs...


In [6]:
from typing import Annotated, TypedDict, Required, get_type_hints

class MaxTokens:
    def __init__(self, max_val: int): self.max_val = max_val

class AgentConfig(TypedDict, total=False):
    llm: Required[str]
    max_tokens: Required[Annotated[int, MaxTokens(4096)]]  # Attach runtime constraint
    temperature: float

def create_agent(cfg: AgentConfig):
    hints = get_type_hints(AgentConfig, include_extras=True)
    # Unwrap the Required to access the inner Annotated type
    inner_type = hints["max_tokens"].__args__[0]
    token_meta = next(m for m in inner_type.__metadata__ if isinstance(m, MaxTokens))
    if cfg["max_tokens"] > token_meta.max_val:
        raise ValueError(f"max_tokens cannot exceed {token_meta.max_val}")
    print(f"Created agent with {cfg['llm']} | max_tokens={cfg['max_tokens']}")

create_agent({"llm": "gemini-1.5-pro", "max_tokens": 2048, "temperature": 0.2})


Created agent with gemini-1.5-pro | max_tokens=2048


In [5]:
from typing import Literal

Model = Literal["gpt-4", "gemini-1.5-pro", "llama-3-70b"]

def create_agent(model: Model):
    print(f"Using {model}")

create_agent("gemini-1.5-pro")  
# create_agent("bert-base") 

Using gemini-1.5-pro


In [8]:
from typing import TypedDict, NotRequired, Required

class TrainingConfig(TypedDict, total=False):
    model: Required[str]       # Required key
    epochs: Required[int]
    learning_rate: NotRequired[float]  # Optional
    optimizer: NotRequired[str]

cfg: TrainingConfig = {"model": "EfficientNetV2", "epochs": 10}
print(cfg["model"])  # EfficientNetV2


EfficientNetV2


In [9]:
from typing import Generic, TypeVar

T = TypeVar("T")

class DataBatch(Generic[T]):
    def __init__(self, data: list[T]): self.data = data
    def first(self) -> T: return self.data[0]

batch = DataBatch[int]([1, 2, 3])
print(batch.first())  # 1


1


In [ ]:
from typing import Annotated

class Range:
    def __init__(self, min, max): self.min, self.max = min, max

LearningRate = Annotated[float, Range(1e-6, 1e-1)]

def train(lr: LearningRate):
    if not (1e-6 <= lr <= 1e-1):
        raise ValueError("Learning rate out of range")
    print(f"Training with lr={lr}")

train(0.001)  


Training with lr=0.001


In [ ]:
from typing import Protocol

class Retriever(Protocol):
    def retrieve(self, query: str) -> list[str]: ...

class SimpleRetriever:
    def retrieve(self, query: str) -> list[str]:
        return [f"Result for {query}"]

def run_search(r: Retriever):
    print(r.retrieve("LangGraph"))

run_search(SimpleRetriever()) 


['Result for LangGraph']


In [12]:
from typing import Union

def normalize_text(x: Union[str, list[str]]) -> list[str]:
    return [x] if isinstance(x, str) else x

print(normalize_text("AI"))         # ['AI']
print(normalize_text(["AI", "ML"])) # ['AI', 'ML']


['AI']
['AI', 'ML']


In [13]:
from typing import Optional

def get_embedding(text: str, model: Optional[str] = None) -> list[float]:
    model = model or "text-embedding-ada-002"
    return [0.1, 0.2, 0.3]

print(get_embedding("Deep learning"))  # Uses default model


[0.1, 0.2, 0.3]


In [14]:
from typing import cast, Any

raw: Any = {"embedding": [0.1, 0.2, 0.3]}
embedding = cast(list[float], raw["embedding"])
print(sum(embedding))


0.6


In [15]:
from typing import Callable

def run_pipeline(step: Callable[[str], str], data: str) -> str:
    return step(data)

def lowercase(text: str) -> str:
    return text.lower()

print(run_pipeline(lowercase, "HELLO"))  # hello


hello
